# Modélisation - Prédiction LFI 2027

Projet fil rouge B3 DATA IA

Dans ce notebook on essaie de prédire le score de Mélenchon en 2027 à partir des données de 2012, 2017 et 2022. On utilise une régression linéaire pour la tendance, puis on construit des scénarios (union à gauche, participation, etc.).

Plan :
1. Préparation des données
2. Régression linéaire (tendance)
3. Scénarios pour 2027
4. Résultats par région
5. Conclusion

## 1. Préparation des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

rouge = '#E53935'

In [ ]:
# Scores nationaux de Melenchon
annees = np.array([2012, 2017, 2022])
scores = np.array([11.10, 19.58, 21.95])

plt.plot(annees, scores, 'o-', color=rouge)
plt.title('Score de Mélenchon')
plt.ylabel('%')
plt.show()

## 2. Régression linéaire

On ajuste une droite sur les 3 points pour estimer la tendance et prédire 2027.

Attention : avec seulement 3 points c'est une estimation très simple, à prendre avec des pincettes. La courbe ralentit déjà entre 2017 et 2022 donc la droite va sûrement surestimer un peu.

In [ ]:
model = LinearRegression()
model.fit(annees.reshape(-1, 1), scores)

pred_2027 = model.predict([[2027]])[0]
print('Pente :', round(model.coef_[0], 2), 'point par an')
print('Prédiction 2027 (tendance simple) :', round(pred_2027, 1), '%')
print('R2 :', round(model.score(annees.reshape(-1, 1), scores), 3))

In [ ]:
annees_full = [2012, 2017, 2022, 2027]
scores_full = list(scores) + [pred_2027]

plt.figure(figsize=(8, 5))
plt.plot([2012, 2017, 2022], scores, 'o-', color=rouge, label='Réel')
plt.plot([2022, 2027], [21.95, pred_2027], 'o--', color=rouge, alpha=0.6, label='Prédiction')
plt.text(2027, pred_2027 + 0.5, f'{pred_2027:.1f} %', ha='center', fontweight='bold')
plt.title('Tendance et prédiction 2027')
plt.ylabel('% des voix')
plt.xticks(annees_full)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Scénarios pour 2027

La tendance seule ne suffit pas : le score dépend aussi de choix politiques. On construit donc 3 scénarios en partant du score 2022 (21,95 %) et en ajoutant/retirant des points selon les hypothèses.

Les hypothèses (estimées à partir de l'analyse) :
- union à gauche : jusqu'à +6,5 pts
- participation : +0,4 pt de LFI pour chaque +1 % de participation
- mobilisation des jeunes : +0,15 pt pour chaque +1 %
- vote utile si le RN est très haut

In [ ]:
base = 21.95
tendance = 1.5  # progression naturelle

def calcul_score(union, part, jeunes, candidat, vote_utile):
    score = base + tendance + union + part * 0.4 + jeunes * 0.15 + candidat + vote_utile
    return round(max(10, min(score, 45)), 1)

scenarios = {
    'Pessimiste': calcul_score(union=0, part=-2, jeunes=-0.5, candidat=-1, vote_utile=0),
    'Réaliste':   calcul_score(union=2.8, part=1.5, jeunes=0.5, candidat=0, vote_utile=0.5),
    'Optimiste':  calcul_score(union=6.5, part=4, jeunes=1.5, candidat=1.5, vote_utile=1.5),
}

for nom, val in scenarios.items():
    qualif = 'qualifié' if val >= 18 else 'incertain'
    print(f'{nom:12s} : {val} %  ({qualif} pour le 2e tour)')

In [ ]:
plt.figure(figsize=(8, 5))
couleurs = ['#B71C1C', rouge, '#FF8A65']
plt.bar(list(scenarios.keys()), list(scenarios.values()), color=couleurs)
for i, v in enumerate(scenarios.values()):
    plt.text(i, v + 0.3, f'{v} %', ha='center', fontweight='bold')
plt.axhline(18, color='orange', linestyle='--', label='Seuil 2e tour (~18 %)')
plt.axhline(21.95, color='grey', linestyle=':', label='Score 2022')
plt.ylabel('% des voix')
plt.title('Scénarios LFI pour 2027')
plt.legend()
plt.show()

## Signal de mobilisation : les européennes 2024

On ne met pas les européennes dans la régression (une européenne ne se compare pas à une présidentielle). Par contre, l'écart entre les deux est un bon **indicateur de mobilisation** : il mesure le réservoir d'électeurs LFI qui votent à la présidentielle mais se démobilisent aux scrutins secondaires. Plus ce réservoir est grand, plus une bonne campagne de mobilisation peut rapporter.

In [ ]:
# score LFI a la presidentielle 2022 vs aux europeennes 2024, par region
pres = pd.read_csv('data_clean/elections_2022_clean.csv', sep=';', encoding='utf-8-sig')
pres['code'] = pres['Code_region'].astype(str).str.zfill(2)
pres['pct_pres'] = pres['Melenchon'] / pres['Exprimés'] * 100

euro = pd.read_csv('data_clean/europeennes_2024_clean.csv', sep=';', encoding='utf-8-sig')
euro['code'] = euro['Code_region'].astype(str).str.zfill(2)

metro = ['11', '24', '27', '28', '32', '44', '52', '53', '75', '76', '84', '93', '94']
comp = pres[pres['code'].isin(metro)].merge(euro[['code', '% LFI']], on='code', how='inner')
comp['reservoir'] = (comp['pct_pres'] - comp['% LFI']).round(1)
comp = comp.sort_values('reservoir', ascending=False)

print('Reservoir de mobilisation par region (presidentielle 2022 - europeennes 2024) :')
print(comp[['Region', 'pct_pres', '% LFI', 'reservoir']].round(1).to_string(index=False))
print()
print('Reservoir moyen :', round(comp['reservoir'].mean(), 1), 'points')
print('-> Partout LFI perd ~10 a 14 points entre les deux scrutins : electorat tres intermittent.')

## 4. Résultats par région

On applique le scénario réaliste à chaque région, en partant de son score 2022.

In [ ]:
df_2022 = pd.read_csv('data_clean/elections_2022_clean.csv', sep=';', encoding='utf-8-sig')
df_2022['pct_mel'] = df_2022['Melenchon'] / df_2022['Exprimés'] * 100

# metropole seulement
df_metro = df_2022[df_2022['Code_region'].astype(str).str.len() == 2].copy()

# scenario realiste : +2.8 (union) +0.6 (part) +0.075 (jeunes) +0.5 (vote utile) +1.5 (tendance)
ajout = 2.8 + 0.6 + 0.075 + 0.5 + 1.5
df_metro['proj_2027'] = (df_metro['pct_mel'] + ajout).round(1)

df_metro = df_metro.sort_values('proj_2027', ascending=False)
print(df_metro[['Region', 'pct_mel', 'proj_2027']].round(1).to_string(index=False))

In [ ]:
df_plot = df_metro.sort_values('pct_mel')
plt.figure(figsize=(10, 6))
plt.barh(df_plot['Region'], df_plot['pct_mel'], color='#FFCDD2', label='2022')
plt.barh(df_plot['Region'], df_plot['proj_2027'] - df_plot['pct_mel'],
         left=df_plot['pct_mel'], color=rouge, label='Gain projeté 2027')
plt.xlabel('% des voix')
plt.title('Projection 2027 par région (scénario réaliste)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Conclusion

- La régression linéaire donne une tendance autour de 26 % mais elle surestime (la courbe ralentit déjà).
- Les scénarios sont plus parlants :
  - pessimiste ~18 % (qualification incertaine)
  - réaliste ~22-24 % (probable)
  - optimiste ~26-28 % (très probable)
- Le facteur le plus important est l'union à gauche (jusqu'à +6,5 pts).

Le simulateur interactif dans l'application Streamlit permet de tester d'autres combinaisons de paramètres.